# VM1 후속 — hq 배치(≈17:30)와 Can calql 사전학습이 끝난 뒤 붙이는 셀 (동시 VM 3대 기준)
환경은 이미 있으므로 설치 셀 없음. 기존 VM1 노트북에 셀을 복사해 붙이거나, 이 노트북을 열어 "연결 ▾ → 활성 런타임(VM1)"에 붙인다.
1. keepalive 정지 → 2. 상태 확인 → 3. Square td 사전학습 ×3 + Can calql 온라인 ×3 → 4. keepalive → (≈20:00, VM3 사전학습 끝나면) 5. Square calql 온라인 ×3 → 4. keepalive.

## 1. 상태 확인 — hq 7개 `[done]`, `calql_can_s{1,2,3}.pt` 존재

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
for E in square_tent12_hq_s1 square_tent12_hq_s2 square_tent12_hq_s3 square_tent12_s4 square_tent12_s5 square_baseline_s4 square_baseline_s5; do
  echo "$E: $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80)"; done
ls -lh $PROJ/logs/pretrain/calql_can_s*.pt 2>/dev/null
for S in 1 2 3; do echo "calql s$S: $(grep '^\[done\]\|Error\|Traceback' $PROJ/logs/pretrain_cql_calql_s$S.out | tail -n 1)"; done
echo "processes: $(ps aux | grep -c '[t]rain_dsrl.py\|[o]ffline_pretrain.py')"; free -g | head -2

## 2. Square td 사전학습 ×3 (≈1.2 h, π_dp 1회/step) + Can calql 온라인 ×3 (150k). RAM ≈ 3×17 + 3×6 GB

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
PROJ=/content/drive/MyDrive/dsrl_project; mkdir -p $PROJ/logs/pretrain
test -f $PROJ/offline/square_train_offline.npz || { echo "square npz 없음 (VM3 9번 먼저)"; exit 1; }
DS="--config-path=cfg/robomimic --config-name=dsrl_square.yaml offline_data_path=$PROJ/offline/square_train_offline.npz log_dir=$PROJ/logs"
for SEED in 1 2 3; do
  nohup python offline_pretrain.py $DS seed=$SEED pretrain.method=cql pretrain.cql_alpha=0 pretrain.out_path=$PROJ/logs/pretrain/td_square_s$SEED.pt > $PROJ/logs/pretrain_sq_td_s$SEED.out 2>&1 &
  echo "started square td seed $SEED (pid $!)"
done
CFG="--config-path=cfg/robomimic --config-name=dsrl_can.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
for S in 1 2 3; do
  test -f $PROJ/logs/pretrain/calql_can_s$S.pt || { echo "calql_can_s$S.pt 없음"; continue; }
  nohup python train_dsrl.py $CFG exp_id=can_calql_s$S seed=$S variant=calql pretrain_path=$PROJ/logs/pretrain/calql_can_s$S.pt $COMMON > $PROJ/logs/can_calql_s$S.out 2>&1 &
  echo "started can_calql_s$S (pid $!)"
done
sleep 120; for S in 1 2 3; do grep "\[pretrain\]\|\[eval\]\|Error\|Traceback" $PROJ/logs/can_calql_s$S.out | tail -n 2; done; free -g | head -2

## 3. keepalive (반납 없음)

In [ ]:
import subprocess, time
PROJ = '/content/drive/MyDrive/dsrl_project'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while True:
    running = sh("ps aux | grep '[o]ffline_pretrain.py\\|[t]rain_dsrl.py' | grep -o 'exp_id=[a-z_0-9]*\\|pretrain.method=[a-z]*\\|seed=[0-9]*' | tr '\\n' ' '")
    ram = sh("free -g | awk 'NR==2{print $3\"/\"$2}'")
    prog = sh("for f in $(ls -t %s/logs/pretrain_*.out %s/logs/can_*.out %s/logs/square_*.out 2>/dev/null | head -n 12); do "
              "n=$(basename $f .out); l=$(grep '^\\[cql\\]\\|^\\[calql\\]\\|^\\[distill\\]\\|\\[eval\\]\\|\\[done\\]' $f | tail -n 1 | cut -c1-70); "
              "echo -n \"$n: $l | \"; done" % (PROJ, PROJ, PROJ))
    print(time.strftime('%H:%M'), 'ram', ram, '|', running or '(none running)', '|', prog, flush=True)
    if not running:
        print('nothing running (VM kept)', flush=True)
        break
    time.sleep(600)

## 5. (≈20:00, VM3의 `calql_square_s*.pt`가 생기면) Square calql 온라인 ×3, 100k. RAM: Can calql 3 + Square calql 3 = 6 run ≈ 102 GB

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_square.yaml"
COMMON="log_dir=$PROJ/logs train.total_env_steps=100000 offline_mix.mode=none load_offline_data=False"
for S in 1 2 3; do
  test -f $PROJ/logs/pretrain/calql_square_s$S.pt || { echo "calql_square_s$S.pt 아직 없음"; continue; }
  nohup python train_dsrl.py $CFG exp_id=square_calql_s$S seed=$S variant=calql pretrain_path=$PROJ/logs/pretrain/calql_square_s$S.pt $COMMON > $PROJ/logs/square_calql_s$S.out 2>&1 &
  echo "started square_calql_s$S (pid $!)"
done
sleep 120; for S in 1 2 3; do grep "\[pretrain\]\|\[eval\]\|Error\|Traceback" $PROJ/logs/square_calql_s$S.out 2>/dev/null | tail -n 2; done; free -g | head -2